In [134]:
%pip install --upgrade --quiet yandex-cloud-ml-sdk

I0000 00:00:1745915434.719670  156268 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


Note: you may need to restart the kernel to use updated packages.


In [129]:
from IPython.display import Markdown, display
import os
from yandex_cloud_ml_sdk import YCloudML
from glob import glob
from tqdm.auto import tqdm
import pandas as pd
from yandex_cloud_ml_sdk.search_indexes import (
    StaticIndexChunkingStrategy,
    HybridSearchIndexType,
    ReciprocalRankFusionIndexCombinationStrategy,
)


def printx(string):
    display(Markdown(string))

folder_id = 'b1gst3c7cskk2big5fqn'
api_key = 'AQVNzzJielnSayrAOlQWlxDMK49OShvzdqtUQdAp'

sdk = YCloudML(folder_id=folder_id, auth=api_key)
model = sdk.models.completions("yandexgpt", model_version="rc")

In [130]:
def create_thread():
    return sdk.threads.create(ttl_days=1, expiration_policy="static")

def create_assistant(model, tools=None):
    kwargs = {}
    if tools and len(tools) > 0:
        kwargs = {"tools": tools}
    return sdk.assistants.create(
        model, ttl_days=1, expiration_policy="since_last_active", **kwargs
    )

def get_token_count(text):
    return len(model.tokenize(text))

def upload_file():
    return sdk.files.upload('/Users/ogzeus/Downloads/data2021.json', ttl_days=1, expiration_policy="static")

In [138]:
from pydantic import BaseModel, Field
from typing import Optional

class GetAdmissionDeadlineParams(BaseModel):
    """Функция возвращает крайний срок подачи документов в МАИ."""

def get_admission_deadline(**kwargs):
    def process(self, thread):
        return "Крайний срок подачи документов в МАИ — 25 июля 2025 года."
    return process()

class HelloWorldBot(BaseModel):
    '''Пишет приветственные слова'''
def get_hello_world(**kwargs):
    class ToolResponse:
        def process(self, thread):
            return 'Привет, я Ассистент-бот МАИ, который может ответить на любые твои вопросы про поступление МАИ. Если ты понимаешь, что я не отвечаю на твои сообщения, то вызови оператора, нажав на кнопку "Оператор" или напиши в чат "Позови оператора"'
    return ToolResponse()

class CallOperator(BaseModel):
    '''Вызывает оператор'''
def get_call_operator(**kwargs):
    def process(self, thread):
        return 'вызов оператора'
    return process(thread)

class Agent:
    def __init__(self, assistant=None, instruction=None, search_index=None, tools=None):
        self.thread = None
        if assistant:
            self.assistant = assistant
        else:
            if tools:
                self.tools = {x['name']: x['fn'] for x in tools}
                tool_defs = [sdk.tools.function(x['model']) for x in tools]
            else:
                self.tools = {}
                tool_defs = []
            if search_index:
                tool_defs.append(sdk.tools.search_index(search_index))
            self.assistant = create_assistant(model, tool_defs)
        if instruction:
            self.assistant.update(instruction=instruction)

    def get_thread(self, thread=None):
        if thread:
            return thread
        if self.thread is None:
            self.thread = create_thread()
        return self.thread

    def __call__(self, message, thread=None):
        thread = self.get_thread(thread)
        thread.write(message)
        run = self.assistant.run(thread)
        res = run.wait()
        if res.tool_calls:
            result = []
            for f in res.tool_calls:
                print(f" + Вызов функции: {f.function.name}, args={f.function.arguments}")
                fn = self.tools[f.function.name]
                obj = fn(**f.function.arguments)
                x = obj.process(thread)
                result.append({"name": f.function.name, "content": x})
            run.submit_tool_results(result)
            res = run.wait()
        return res.text

    def restart(self):
        if self.thread:
            self.thread.delete()
            self.thread = sdk.threads.create(name="Test", ttl_days=1, expiration_policy="static")

    def done(self, delete_assistant=False):
        if self.thread:
            self.thread.delete()
        if delete_assistant:
            self.assistant.delete()


In [139]:
g = upload_file()

In [133]:
g

File(id='fvt5mgpkhv71vmcr06o8', expiration_config=ExpirationConfig(ttl_days=1, expiration_policy=<ExpirationPolicy.STATIC: 1>), name=None, description=None, mime_type='text/plain', created_by='aje0hf7fque78sn3gnkn', created_at=datetime.datetime(2025, 5, 1, 8, 9, 12, 706085), updated_by='aje0hf7fque78sn3gnkn', updated_at=datetime.datetime(2025, 5, 1, 8, 9, 12, 706085), expires_at=datetime.datetime(2025, 5, 2, 8, 9, 12, 706085), labels=None)

In [134]:
op = sdk.search_indexes.create_deferred(
    g,
    index_type=HybridSearchIndexType(
        chunking_strategy=StaticIndexChunkingStrategy(
            max_chunk_size_tokens=1000, chunk_overlap_tokens=100
        ),
        combination_strategy=ReciprocalRankFusionIndexCombinationStrategy(),
    )
)
index = op.wait()

In [140]:
agent = Agent(
    instruction=instruction,
    search_index=index,
    tools=[{
        "name": "GetAdmissionDeadlineParams",
        "model": GetAdmissionDeadlineParams,
        "fn": get_admission_deadline
    }]
)

response = agent("вызови поддержку")
printx(response)

TypeError: Function call parameters could be only jsonschema dict, pydantuc model class or pydantic dataclass

In [ ]:
agent("вызови оператора")

In [126]:
!pip install ragas

I0000 00:00:1746063699.123056 2224848 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 752.2 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 3.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 562.8 kB/s eta 0:00:00:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.3/661.3 kB 2.1 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 2.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.2/437.2 kB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 3.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0

In [127]:
from ragas import EvaluationDataset

eval_dataset = load_dataset("explodinggradients/earning_report_summary",split="train")

NameError: name 'load_dataset' is not defined